# Check-in 4 GPU runner (C2 + strength sweeps)

Runtime must be **GPU** (L4/A100). Then **Runtime → Run all**.

Installs MarkDiffusion, mounts Drive, stages Check-in 3 keys/images,
runs offline analysis/figures, then C2 imprint (SD 1.5) and crop/purification sweeps.


In [ ]:
# 0) Device check
import torch
print('cuda:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Switch runtime to GPU first (Runtime → Change runtime type)'


In [ ]:
# 1) Mount Drive (optional for offline analysis; required for C2/sweeps keys)
from pathlib import Path
DRIVE_OK = False
try:
  from google.colab import drive
  drive.mount('/content/drive', force_remount=False)
  DRIVE_OK = Path('/content/drive').exists()
  print('Drive mounted:', DRIVE_OK)
except Exception as e:
  print('Drive mount skipped/failed:', e)
  print('Offline analysis can still run from the cloned repo measurements.')


In [ ]:
# 2) Clone / update repo
import pathlib, subprocess
REPO = pathlib.Path('/content/cmpe261-watermark-forgery')
if not (REPO / '.git').exists():
    subprocess.check_call(['git', 'clone', 'https://github.com/yashashav-dk/cmpe261-watermark-forgery.git', str(REPO)])
else:
    subprocess.check_call(['git', '-C', str(REPO), 'pull', '--ff-only'])
%cd /content/cmpe261-watermark-forgery
print('repo at', REPO)


In [ ]:
# 3) Install wmforge + pinned MarkDiffusion, then repair torch/torchvision match
# Colab often ends with a broken torchvision after pip resolves [attack]; that
# shows up later as CLIPImageProcessor / torchvision.extension import errors.
import subprocess, sys
MARK = '9d81656d1a5f9e5194fc2f727bb795ef29e53809'
subprocess.check_call([
  sys.executable, '-m', 'pip', 'install', '-q',
  '-e', '.[attack]',
  f'markdiffusion @ git+https://github.com/THU-BPM/MarkDiffusion@{MARK}',
  'qrcode', 'galois', 'ldpc<2', 'matplotlib', 'pandas', 'markdown',
  'diffusers==0.40.0',
])
# Drop mismatched audio wheel; force the Check-in 3 CUDA pair that worked on L4/A100.
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio'])
subprocess.check_call([
  sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
  'torch==2.10.0', 'torchvision==0.25.0',
  '--index-url', 'https://download.pytorch.org/whl/cu128',
])
import torch, torchvision, diffusers, transformers, markdiffusion
from transformers import CLIPImageProcessor
from diffusers import StableDiffusionPipeline
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('diffusers', diffusers.__version__, '| transformers', transformers.__version__)
print('cuda', torch.cuda.is_available(), '| CLIPImageProcessor OK | SD pipeline import OK')
print('markdiffusion ok')


In [ ]:
# 4) Locate Check-in 3 full tree (keys + images) on Drive and stage it
from pathlib import Path
import shutil

checkin3 = None
drive_root = Path('/content/drive')
if drive_root.exists():
  candidates = list(drive_root.rglob('measurements-v2.json'))
  print('found measurement logs on Drive:', len(candidates))
  for pth in candidates[:20]:
      print(' -', pth)
  for pth in candidates:
      root = pth.parent
      if (root / 'keys').exists() and (root / 'images').exists():
          checkin3 = root
          break

dest = Path('results/checkin3/full')
dest.mkdir(parents=True, exist_ok=True)
if checkin3 is None:
  print('No Drive keys/images found. Keeping repo measurements for offline analysis.')
  print('C2 imprint / sweeps will stop with a clear preflight error until keys/images are on Drive.')
else:
  print('CHECKIN3 =', checkin3)
  print('keys:', sorted(x.name for x in (checkin3 / 'keys').glob('*.pkl'))[:8])
  for name in ['measurements-v2.json', 'manifest.json']:
      src = checkin3 / name
      if src.exists():
          shutil.copy2(src, dest / name)
  for cal in checkin3.glob('*-calibration.json'):
      shutil.copy2(cal, dest / cal.name)
  for sub in ['keys', 'images', 'inputs', 'checkpoints']:
      src = checkin3 / sub
      dst = dest / sub
      if src.exists():
          if dst.is_symlink() or dst.exists():
              if dst.is_symlink() or dst.is_file():
                  dst.unlink()
              else:
                  shutil.rmtree(dst)
          dst.symlink_to(src, target_is_directory=True)
          print('linked', sub, '->', src)
print('staged', dest)


In [ ]:
# 5) Offline Check-in 4 analysis + figures (CPU, fast)
import os, subprocess, sys
env = {**os.environ, 'PYTHONPATH': 'src'}
subprocess.check_call([sys.executable, 'scripts/checkin4_analysis.py'], env=env)
subprocess.check_call([sys.executable, 'scripts/checkin4_figures.py'], env=env)
print('offline analysis/figures done')


In [ ]:
# 6) C2 imprint (SD 1.5 attacker) — long GPU job
import os, subprocess, sys
from pathlib import Path
env = {**os.environ, 'PYTHONPATH': 'src'}
keys = list(Path('results/checkin3/full/keys').glob('*.pkl')) if Path('results/checkin3/full/keys').exists() else []
if not keys:
  print('SKIP C2: Check-in 3 keys/images not staged. Upload results/checkin3/full/{keys,images,inputs} to Drive and re-run.')
else:
  cmd = [sys.executable, '-m', 'wmforge.checkin4', 'c2-imprint',
         '--checkin3', 'results/checkin3/full',
         '--out', 'results/checkin4/c2',
         '--device', 'cuda',
         '--schemes', 'GS', 'PRC', 'WIND', 'RI', 'SFW', 'TR']
  print('Running:', ' '.join(cmd))
  subprocess.check_call(cmd, env=env)
  print('C2 imprint finished')


In [ ]:
# 7) Strength sweeps (crop + purification)
import os, subprocess, sys
from pathlib import Path
env = {**os.environ, 'PYTHONPATH': 'src'}
keys = list(Path('results/checkin3/full/keys').glob('*.pkl')) if Path('results/checkin3/full/keys').exists() else []
imgs = Path('results/checkin3/full/images')
if not keys or not imgs.exists():
  print('SKIP sweeps: need Check-in 3 keys/ and images/ staged from Drive.')
else:
  cmd = [sys.executable, '-m', 'wmforge.checkin4', 'sweep',
         '--checkin3', 'results/checkin3/full',
         '--out', 'results/checkin4/sweeps',
         '--device', 'cuda',
         '--schemes', 'GS', 'PRC', 'WIND', 'RI', 'SFW', 'TR',
         '--attacks', 'crop', 'purification']
  print('Running:', ' '.join(cmd))
  subprocess.check_call(cmd, env=env)
  print('Sweeps finished')


In [ ]:
# 8) Zip results for download / Drive copy
import shutil, pathlib
out = pathlib.Path('/content/checkin4_gpu_results')
if out.exists():
    shutil.rmtree(out)
out.mkdir()
for p in ['results/checkin4/c2', 'results/checkin4/sweeps', 'results/checkin4/analysis', 'results/checkin4/figures']:
    src = pathlib.Path(p)
    if src.exists():
        shutil.copytree(src, out / src.name)
zip_path = shutil.make_archive('/content/checkin4_gpu_results', 'zip', out)
print('Download this zip:', zip_path)
drive_out = pathlib.Path('/content/drive/MyDrive/CMPE261_CheckIn4_GPU_Results')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, drive_out / 'checkin4_gpu_results.zip')
print('Copied to', drive_out)
